# धडा 10 - उत्पादनातील एआय एजंट्स

या धड्यात तुम्ही मायक्रोसॉफ्ट एजंट फ्रेमवर्क (पायथन) वापरून AI एजंट्ससाठी **उत्पादन पॅटर्न** शिकाल. आम्ही यावर चर्चा करू:

- **निरीक्षणक्षमता** — एजंट संवादांवर वेळ आणि लॉगिंग जोडणे
- **मूल्यमापन** — प्रतिसाद गुणवत्तेचे स्कोअर करण्यासाठी एक मूल्यांकन एजंट वापरणे
- **खर्च व्यवस्थापन** — टोकन ऑप्टिमायझेशन आणि मॉडेल निवडीसाठी धोरणे

परिस्थिती म्हणजे एक **प्रवास एजंट** जो वापरकर्त्यांना सहलीचे नियोजन करण्यात मदत करतो, ज्यावर निरीक्षण आणि मूल्यांकन परत आहेत.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचार

एआय एजंट्सना प्रोटोटायप्समधून उत्पादनासाठी हलवण्यासाठी तीन मूल स्तंभांवर काळजीपूर्वक लक्ष देणे आवश्यक आहे:

1. **व्हिजिबिलिटी** — तुम्हाला एजंट काय करत आहे, किती वेळ लागतो आणि कोणती साधने कॉल करत आहे याची माहिती पाहिजे. ट्रेसिंग आणि लॉगिंगशिवाय, उत्पादनात येणाऱ्या समस्यांचे निराकरण करणे जवळजवळ अशक्य आहे.

2. **मूल्यमापन** — स्वयंचलित गुणवत्ता तपासण्या सुनिश्चित करतात की एजंटची उत्तरे काळानुसार अचूक, पूर्ण आणि उपयुक्त राहतील. एक मूल्यमापक एजंट परिभाषित निकषांनुसार उत्तरे गुणांकन करू शकतो.

3. **खर्च व्यवस्थापन** — टोकन वापर खर्चावर थेट परिणाम करतो. प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड, आणि कॅशिंग सारख्या धोरणांनी खर्च नियंत्रणात ठेवता येतो, गुणवत्तेवर तडजोड न करता.


## निरीक्षणीय एजंट तयार करणे

आपण प्रवास साधनांची व्याख्या करतो आणि एजंट कॉलला वेळ मोजणीसह घालून लेटन्सीवर निरीक्षण करू शकतो. उत्पादनात तुम्ही OpenTelemetry किंवा तत्सम ट्रेसिंग बॅकएंडसह एकत्रिकरण कराल.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यमापन नमुने

एक सामान्य उत्पादन नमुना म्हणजे दुसरा एजंट **मूल्यमापक** म्हणून वापरणे. मूल्यमापक प्राथमिक एजंटच्या प्रतिसादाचे पूर्णता, अचूकता, आणि उपयुक्तता या पूर्वनिर्धारित निकषांनुसार स्कोअर करतो.

हे सक्षम करते:
- प्रतिसाद वापरकर्त्यांपर्यंत पोहचण्याअगोदर स्वयंचलित गुणवत्ता दरवाजे
- प्रॉम्प्ट किंवा मॉडेल बदलल्यावर पुनरावृत्ती शोधणे
- एजंटच्या कामगिरीचे सातत्याने निरीक्षण करणे


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## खर्च नियंत्रण धोरणे

उत्पादन एआय एजंट्ससाठी खर्च नियंत्रण अत्यंत महत्त्वाचे आहे. येथे काही प्रमुख धोरणे आहेत:

| धोरण | वर्णन |
|---|---|
| **प्रॉम्प्ट ऑप्टिमायझेशन** | सिस्टम सूचना संक्षिप्त ठेवा. इनपुट टोकन्स कमी करण्यासाठी अनावश्यक संदर्भ काढा. |
    "| **मॉडेल निवड** | वर्गीकरण किंवा एक्स्ट्रॅक्शन सारख्या सोप्या कार्यांसाठी लहान, स्वस्त मॉडेल्स (उदा. GPT-5-mini) वापरा, आणि जटिल विचारसरणीसाठी मोठ्या मॉडेल्स राखून ठेवा. |\n",
| **कॅशिंग** | टूल परिणाम व वारंवार विचारल्या जाणाऱ्या प्रश्नांचे परिणाम कॅश करा जेणेकरून अनावश्यक API कॉल टाळता येतील. |
| **टोकन बजेट** | अनपेक्षितपणे लांब उत्तर येण्यापासून प्रतिबंध करण्यासाठी `max_tokens` मर्यादा ठेवा. |
| **बॅचिंग** | शक्य असल्यास, अनेक वापरकर्ता प्रश्न एकाच API कॉलमध्ये एकत्र करा. |

प्रत्यक्षात, एक स्तरांतरित दृष्टिकोन चांगला काम करतो: सोप्या विनंत्यांना वेगवान, स्वस्त मॉडेलकडे मार्गदर्शित करा आणि फक्त जटिल प्रश्नांसाठी अधिक सक्षम मॉडेलला वाढवा.


## सारांश

या धड्यात तुम्ही शिकले कसे करायचे:

1. **एजंट संवादांमध्ये निरीक्षणक्षमता वाढवणे** वेळ नोंदणी आणि लॉगिंगसह, ज्यामुळे ट्रेसिंग आणि मॉनिटरिंगसाठी पाया तयार होतो.
2. **एजंट प्रतिसादांचे मूल्यांकन** आपोआप एका मूल्यांकन एजंटचा वापर करून जो पूर्णता, अचूकता आणि उपयुक्ततेचे गुणांकन करतो.
3. **खर्च व्यवस्थापन** प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड, कॅशिंग आणि टोकन बजेटद्वारे.

हे उत्पादन नमुने तुमचे AI एजंट विश्वसनीय, मोजता येण्याजोगे, आणि प्रमाणात किफायतशीर असल्याचे सुनिश्चित करतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
